In [ ]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
import shutil
from PIL import Image
import json
from tqdm import tqdm
import datasets
datasets.logging.set_verbosity_debug()
from datasets import load_dataset

complete_preference_pairs_num_start = 400000
complete_preference_pairs_num_end = 402000
base_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/hpdv2"
preference_pairs_dir = os.path.join(base_dir, f"preference_pairs-start_{complete_preference_pairs_num_start}-end_{complete_preference_pairs_num_end}")
os.makedirs(preference_pairs_dir, exist_ok=True)

train_json_path = os.path.join(base_dir, "train.json")
# 直接读取 train.json 文件
with open(train_json_path, "r", encoding="utf-8") as f:
    dataset = json.load(f)
print(f"total samples: {len(dataset)}")

# 1. 确定你的图片存放根目录
# 请务必检查：00000000.jpg 所在的文件夹路径
image_source_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/hpdv2-image_folder"

# 2. 处理指定范围内的所有样本
dataset_iter = iter(dataset)
total_samples = complete_preference_pairs_num_end - complete_preference_pairs_num_start
success_count = 0
error_count = 0

# 跳过前面的样本
for _ in range(complete_preference_pairs_num_start):
    try:
        next(dataset_iter)
    except StopIteration:
        print(f"⚠️ 警告：数据集样本数少于 {complete_preference_pairs_num_start}，无法处理指定范围")
        break

# 处理指定范围内的样本
for idx in tqdm(range(total_samples), desc="处理样本"):
    try:
        sample = next(dataset_iter)
        pair_idx = complete_preference_pairs_num_start + idx
        prompt = sample['prompt']
        paths = sample['image_path']
        prefs = sample['human_preference']
        
        # 根据 [0, 1] 逻辑判断：1 是赢家，0 是输家
        win_idx = 0 if prefs[0] > prefs[1] else 1
        lose_idx = 1 - win_idx
        
        # 创建对应的 pair 文件夹
        pair_dir = os.path.join(preference_pairs_dir, f"train_pair_{pair_idx:06d}")
        os.makedirs(pair_dir, exist_ok=True)
        
        # 处理两张图片
        all_images_exist = True
        for label, idx_in_sample in [("win", win_idx), ("lose", lose_idx)]:
            src_path = os.path.join(image_source_dir, paths[idx_in_sample])
            dst_path = os.path.join(pair_dir, f"{label}_image.jpg")
            
            if os.path.exists(src_path):
                shutil.copy(src_path, dst_path)
                # 验证图片是否损坏
                try:
                    with Image.open(dst_path) as img:
                        img.verify()
                except Exception as img_error:
                    print(f"⚠️ 样本 {pair_idx}: {label} 图片损坏: {img_error}")
                    all_images_exist = False
            else:
                print(f"❌ 样本 {pair_idx}: 找不到 {label} 图片! 路径: {src_path}")
                all_images_exist = False

        # 保存 Caption
        if all_images_exist:
            with open(os.path.join(pair_dir, "caption.txt"), "w") as f:
                f.write(prompt)
            success_count += 1
        else:
            error_count += 1
            # 如果图片有问题，删除这个文件夹
            if os.path.exists(pair_dir):
                shutil.rmtree(pair_dir)
    
    except StopIteration:
        print(f"\n⚠️ 数据集已到末尾，实际处理了 {idx} 个样本（期望 {total_samples} 个）")
        break
    except Exception as e:
        error_count += 1
        print(f"\n❌ 处理样本 {pair_idx} 时发生错误: {e}")
        continue

print(f"\n✅ 处理完成！")
print(f"   成功: {success_count} 个样本")
print(f"   失败: {error_count} 个样本")
print(f"   输出目录: {preference_pairs_dir}")

In [ ]:
import os
from IPython.display import display
from PIL import Image as PILImage
import matplotlib.pyplot as plt

pairs_id = 400201
image_type = "win_image"
complete_preference_pairs_num_start = 400000
complete_preference_pairs_num_end = 402000

base_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/hpdv2"
preference_pairs_dir = os.path.join(base_dir, f"preference_pairs-start_{complete_preference_pairs_num_start}-end_{complete_preference_pairs_num_end}")

image_path = os.path.join(preference_pairs_dir, f"train_pair_{pairs_id:06d}", f"{image_type}.jpg")
caption_path = os.path.join(preference_pairs_dir, f"train_pair_{pairs_id:06d}", "caption.txt")

# 显示图像（resize 到 512x512）
if os.path.exists(image_path):
    print(f"图像路径: {image_path}")
    # 打开图像并 resize 到 512x512
    img = PILImage.open(image_path)
    original_size = img.size
    print(f"原始尺寸: {original_size[0]}x{original_size[1]}")
    
    # Resize 到 512x512
    img_resized = img.resize((512, 512), PILImage.Resampling.LANCZOS)
    print(f"Resize 后尺寸: 512x512")
    
    # 直接显示 resize 后的图像
    plt.figure(figsize=(8, 8))
    plt.imshow(img_resized)
    plt.axis('off')
    plt.show()
else:
    print(f"❌ 图像文件不存在: {image_path}")

# 读取并输出 caption.txt 内容
if os.path.exists(caption_path):
    print(f"\nCaption 路径: {caption_path}")
    print("\n" + "="*50)
    print("Caption 内容:")
    print("="*50)
    with open(caption_path, "r", encoding="utf-8") as f:
        caption_content = f.read()
        print(caption_content)
    print("="*50)
else:
    print(f"❌ Caption 文件不存在: {caption_path}")
